# Tầng 2 — Phân loại 7 lớp tổn thương gan (LLD-MMRI, MRI đa thì)

**Chuẩn bị trước khi chạy:**
1. Settings → **Accelerator = GPU T4** · **Internet = On**
2. **Add Data**: dataset LLD raw (`marcohoang/lld-mmri-dataset`)
3. Chạy tuần tự Cell 1 → 7 (không cần restart giữa chừng). Xong **Save Version**.

In [ ]:
# 1) Clone code
import os, glob
if not os.path.exists('/kaggle/working/HCC-TACE-Assist'):
    !git clone -q https://github.com/hdtruong802/HCC-TACE-Assist.git /kaggle/working/HCC-TACE-Assist
!cd /kaggle/working/HCC-TACE-Assist && git pull -q && git log --oneline -1

In [ ]:
# 2) Dò dữ liệu LLD + kiểm GPU
ann = glob.glob('/kaggle/input/**/LLD_MMRI_Annotation.json', recursive=True)
assert ann, 'Chưa thấy LLD — Add Data marcohoang/lld-mmri-dataset'
LLD_ROOT = os.path.dirname(ann[0]); print('LLD_ROOT =', LLD_ROOT)
print('images:', os.path.isdir(f'{LLD_ROOT}/images'), '| labels:', os.path.isdir(f'{LLD_ROOT}/labels'))
import torch; print('GPU:', torch.cuda.is_available(),
                     torch.cuda.get_device_name(0) if torch.cuda.is_available() else '<-- BAT GPU trong Settings!')

In [ ]:
# 3) (Tùy chọn) QC ROI từ mask — đã xác nhận rồi, có thể bỏ qua
!cd /kaggle/working/HCC-TACE-Assist && python scripts/qc_lld.py --config configs/data/lld.yaml --data-root "{LLD_ROOT}" --n 7 --phase "C+A" --out /kaggle/working/qc_lld.png
from IPython.display import Image, display
display(Image('/kaggle/working/qc_lld.png'))

In [ ]:
# 4) BUILD cache ROI + manifest (~15-20 phút)
!cd /kaggle/working/HCC-TACE-Assist && python scripts/build_lld.py --config configs/data/lld.yaml --data-root "{LLD_ROOT}" --out-dir /kaggle/working

In [ ]:
# 5) SPLIT patient-level phân tầng 7 lớp + leakage test
!cd /kaggle/working/HCC-TACE-Assist && python scripts/make_split_lld.py --config configs/data/lld.yaml --manifest /kaggle/working/manifest_lld.csv --out-dir /kaggle/working

In [ ]:
# 6) TRAIN classifier 7 lớp (đa thì in_chans=8) — GPU
!cd /kaggle/working/HCC-TACE-Assist && python -m src.training.train_cls --config configs/train/lld_cls.yaml --data-root /kaggle/working --fold 0

In [ ]:
# 7) EVAL: macro-F1 + per-class + confusion + malignant-AUC (có CI)
ARCH='convnextv2_nano'
!cd /kaggle/working/HCC-TACE-Assist && python -m src.evaluation.evaluate_cls --config configs/train/lld_cls.yaml --data-root /kaggle/working --ckpt /kaggle/working/outputs_cls/{ARCH}_fold0/best.ckpt --split val --fold 0
from IPython.display import Image, display
p=f'/kaggle/working/outputs_cls/{ARCH}_fold0/eval_val/confusion.png'
if os.path.exists(p): display(Image(p))

## Ghi chú
- Cell 2 phải in `GPU: True`; nếu `False` → bật GPU rồi chạy lại từ Cell 1.
- Đa thì ghép **8 kênh** (in_chans=8); thì thiếu → kênh 0. `class_weight=auto`, chọn best **macro-F1**.
- **malignant-AUC** = tổng xác suất lớp {1,3,6} (ICC/di căn/HCC).
- Lớp hiếm (FNH 46, di căn 51) → per-class F1 nhiễu, bình thường.
- Tải `outputs_cls/.../eval_val/*` về `image_eval/lld_convnextv2/` để làm report.